# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/imnotparama/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "search_volume"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

test_df = df.iloc[test_idx].copy()
test_df["model_score"] = rf.predict_proba(X.iloc[test_idx])[:, 1]

# Reason codes, combining model score with interpretable rule-based context
def assign_reason_code(row):
    if row["model_score"] >= 0.65:
        return "model_decline_risk"
    elif row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 100:
        return "stale_visible_page"
    elif row["impressions_90d"] >= 500 and row["avg_position"] > 0 and row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return "ctr_review_candidate"
    else:
        return "monitor_only"

test_df["reason_code"] = test_df.apply(assign_reason_code, axis=1)
test_df["action"] = test_df["reason_code"].map({
    "model_decline_risk": "review_for_refresh",
    "stale_visible_page": "review_for_refresh",
    "ctr_review_candidate": "review_metadata_ctr",
    "monitor_only": "monitor"
})

ranked_queue = test_df.sort_values("model_score", ascending=False)[
    ["content_id", "model_score", "reason_code", "action",
     "days_since_last_update", "impressions_90d", "avg_position", "ctr", "trend_direction"]
]
print(ranked_queue.head(20))

Working dir: /content/flyrank-ml-internship
                 content_id  model_score         reason_code  \
18989  content_edf5b03b36aa        0.980  model_decline_risk   
16667  content_720edd4e1491        0.975  model_decline_risk   
14343  content_9ac61c04930e        0.970  model_decline_risk   
7050   content_7a6cb4857fe6        0.970  model_decline_risk   
22526  content_1d0963b56227        0.970  model_decline_risk   
6228   content_e988c1699454        0.965  model_decline_risk   
3329   content_eb3b2c3bbc34        0.965  model_decline_risk   
22730  content_bb1edbf0ba6c        0.960  model_decline_risk   
21023  content_8a4f4f22c861        0.960  model_decline_risk   
12246  content_c5cdd229c348        0.955  model_decline_risk   
2141   content_085de37f9ed6        0.955  model_decline_risk   
12519  content_24aafbba8fd5        0.955  model_decline_risk   
17347  content_d248f78edf97        0.955  model_decline_risk   
16867  content_87cbed1ec54a        0.955  model_decline_risk

**Ranked action queue:** Each page gets a model_score (0-1, Random
Forest probability), a reason_code explaining why it was flagged, and
an action label. Reason codes combine the model's own confidence
(model_decline_risk) with interpretable rule context (stale_visible_page,
ctr_review_candidate) so a human reviewer isn't just trusting an opaque
number — they can see the concrete signal behind each flag.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** This playbook is decision-support for a content or
SEO team lead deciding where to spend limited review bandwidth each
sprint. It ranks candidate pages; it does not make or execute any
content decision itself.

**Limits (careful language throughout):**
- The model's Precision@50 was observed at 0.580 under a grouped-by-
  client validation split (8 held-out clients) — this is directional
  evidence the ranking beats chance (base rate 0.517), not a guarantee
  for any individual page.
- The label (is_declining_label, from trend_direction) is a proxy
  computed from the current window, not a validated future outcome.
- GA4-derived signals are only available for a small, non-random
  subset of pages (per ML-04's finding: ~4.2% of a sample month) — the
  ranking leans more heavily on search-console-style signals as a
  result.
- This is observational, cross-sectional data. Nothing here supports
  "refreshing this page will cause recovery" — only "this page shows
  patterns associated with decline, worth a human look."

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human review required before any action:**
- Confirm the page isn't mid-migration, recently redirected, or
  affected by a known seasonal pattern (per the lane guide's decline-
  vs-consolidation-vs-seasonality checklist).
- Check whether a sibling/related page absorbed the traffic
  (consolidation), rather than genuine decline.
- Verify the page isn't already scheduled for deprecation.

**What should NEVER be automated:**
- Auto-deleting, auto-redirecting, or auto-rewriting content based on
  this score alone.
- Treating "monitor_only" as "safe to ignore permanently" — it means
  "no strong signal yet," not "confirmed healthy."
- Using this ranking as a performance/quality judgment of the content
  creator or team responsible for a page.
- Fully replacing human review with the score, even at high
  model_score — the model's own wrong-case review (ML-08) showed it
  can flag stable/growing pages, so a human check is not optional.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Retrain triggers:**
- If Precision@50 on a fresh, out-of-time validation slice drops
  meaningfully below the observed 0.580 baseline, investigate before
  trusting the ranking further.
- If the client mix changes substantially (new clients onboarded,
  major client churn), re-validate with a grouped split on the updated
  population — client-specific patterns may no longer transfer.
- If GA4 coverage changes significantly (currently ~4.2% in the
  sampled month), reassess whether GA4-derived features should be
  reweighted or excluded.

**Monitoring:** Track reviewer feedback on flagged pages (did the
human agree with the flag?) as a lightweight, ongoing check —
persistent disagreement is itself a retrain/rework signal.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
import os, json

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Export the ranked queue (regenerated each run, stays out of git per CI leak-guard)
ranked_queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)
print(f"Wrote {len(ranked_queue)} rows to work/outputs/action_playbook_queue.csv")

# Export metrics JSON (the "receipts" — this IS committed to git)
metrics = {
    "model": "RandomForestClassifier",
    "validation_split": "grouped_by_client",
    "test_clients": int(df.iloc[test_idx]["client_id"].nunique()),
    "precision_at_50": 0.580,
    "base_rate": float(y.iloc[test_idx].mean()),
    "baseline_rule_precision_at_50": 0.620,
    "random_split_precision_at_50_dishonest": 0.920,
    "note": "Random split score (0.920) is inflated by client leakage; grouped split (0.580) is the honest number."
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/playbook_metrics.json")
print(json.dumps(metrics, indent=2))

Wrote 7115 rows to work/outputs/action_playbook_queue.csv
Wrote work/outputs/playbook_metrics.json
{
  "model": "RandomForestClassifier",
  "validation_split": "grouped_by_client",
  "test_clients": 8,
  "precision_at_50": 0.58,
  "base_rate": 0.516514406184118,
  "baseline_rule_precision_at_50": 0.62,
  "random_split_precision_at_50_dishonest": 0.92,
  "note": "Random split score (0.920) is inflated by client leakage; grouped split (0.580) is the honest number."
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.